# 05 - Trajectories: emotion dynamics across a story's tokens (Q3)

**What this notebook is for.** The third research question: in stories that move through
three emotions (sequentially or simultaneously), how does the per-token cosine between the
residual stream and each emotion's probe direction evolve — gradual ramp-and-crossover, or
steps at lexical cues? This notebook exhibits the collected substrate and the figure
vocabulary. **It draws no verdicts**: the scoring reads for Q3.H1.E1 (ramp-vs-step statistic,
anticipation window, layer contrast, pass bars) are not yet registered in TREE.md, and per the
anti-peeking discipline the full-corpus aggregate stays out of this notebook until they are.
Everything here is single-story exhibition plus descriptive corpus statistics.

**Data lineage (read this first).**
- **Stories**: 5,888 three-emotion stories generated by `google/gemma-4-31b-it` *itself*
  (the self-generated convention, TREE Q1.H2.E10) — 173 emotion triples x 6 permutations x
  2 modes x 3 samples, seed 20260721, temperature 1.3. QC dropped emotion-word leaks (189)
  and tag mismatches (151). HF: `abotresol/emotion-combined-stories-gemma-4-31b-it`.
- **Trajectories**: teacher-forced forward pass through the same model (bf16, right padding —
  post-dates the padding-side instrument fix, TREE Q1.H3.E4), residual captured at layers
  {6, 15, 24, 33, 42, 51}. Each story's shard stores raw per-token dots onto 207 unit probes:
  171 **corpus-lineage** (external 4B-generated stories), 12 **selfgen-lineage** (n=256
  self-generated, the E10 winner — only the 12 battery emotions have this lineage), and 24
  fixed-seed **random** null directions. The two probe lineages are measurably different
  objects (contrast cosine 0.22, TREE Q1.H2.E6) — figures state which one they use.
  HF: `abotresol/emotion-combined-trajectories-gemma-4-31b-it`.
- **Convention**: all cosines below are **centered** (per-story token-mean removed — the E9
  lesson that uncentered readouts bury affect under shared story-reading structure), computed
  from the shards' raw dots and stored centered norms; smoothing windows are stated per figure.


In [1]:
import json

import numpy as np

from emotion_vectors.artifacts import fetch
from emotion_vectors import trajectory_plots as tp
from emotion_vectors.trajectories import transition_windows

manifest = [json.loads(l) for l in fetch("combined_trajectories/manifest.jsonl").read_text().splitlines()]
labels = json.loads(fetch("combined_trajectories/probe_labels.json").read_text())
config = json.loads(fetch("combined_trajectories/run_config.json").read_text())
LAYERS = config["layers"]
print(f"{len(manifest)} stories | layers {LAYERS} | {len(labels)} probes")


5888 stories | layers [6, 15, 24, 33, 42, 51] | 207 probes


## 1. What was collected

Descriptive statistics only — the corpus as measured, before any hypothesis touches it.

In [2]:
from collections import Counter

modes = Counter(m["mode"] for m in manifest)
tokens = np.array([m["n_tokens"] for m in manifest])
cats = Counter(m["category"] for m in manifest)
print(f"modes: {dict(modes)}")
print(f"tokens/story: median {np.median(tokens):.0f}, range {tokens.min()}-{tokens.max()}")
print(f"categories: {dict(cats)}")
seq3 = sum(1 for m in manifest if m["mode"] == "SEQUENTIAL" and len(m["phase_token_starts"]) == 3)
print(f"sequential stories with clean 3-phase alignment: {seq3}/{modes['SEQUENTIAL']}")


modes: {'SIMULTANEOUS': 2878, 'SEQUENTIAL': 3010}
tokens/story: median 210, range 171-365
categories: {'B_conflict': 1644, 'E_arousal_mismatch': 1092, 'F_valence_spread': 1084, 'A_superposition': 993, 'D_timescale': 1075}
sequential stories with clean 3-phase alignment: 3009/3010


## 2. One sequential story, every view — scrubbable across layers

Story `t000_seq_p2_2f9faf62` (upset -> unsettled -> cheerful, 236 tokens) — chosen as the
first mid-length cross-valence sequential story in the manifest, before any scoring existed.
Corpus-lineage probes (the triple's emotions are not all in the 12-emotion selfgen set).
Every figure below carries a **layer slider** over the six captured bands {6, 15, 24, 33, 42, 51}
(the explore_layers.ipynb convention) — the paper's layer story predicts token-level jitter
early and scene-level phase structure mid-late, so drag it and look for exactly that contrast.
The static small-multiples and speed views follow for the print/report path.


In [3]:
STORY = "t000_seq_p2_2f9faf62"
row = next(m for m in manifest if m["story_id"] == STORY)
shard = np.load(fetch(f"combined_trajectories/shards/{STORY}.npz"))
emotions, starts = row["phase_emotions"], row["phase_token_starts"]

from emotion_vectors.interactive import (
    trajectory_heatmap_scrubber,
    trajectory_lines_scrubber,
    trajectory_ternary_scrubber,
)

cos_by_layer = {
    k: tp.smooth(tp.story_cosines(shard, labels, emotions, i, lineage="corpus"), window=8)
    for i, k in enumerate(LAYERS)
}
trajectory_lines_scrubber(
    LAYERS, cos_by_layer, emotions, starts, default_layer=33,
    title="Per-token probe trajectories — drag the slider across layers",
).show()


In [4]:
bary_by_layer = {k: tp.barycentric(cos_by_layer[k]) for k in LAYERS}
trajectory_ternary_scrubber(
    LAYERS, bary_by_layer, emotions, starts, default_layer=33,
    title="Emotion phase space (softmax display transform, T=28) — scrub layers",
).show()


In [5]:
per_layer = [
    tp.smooth(tp.story_cosines(shard, labels, emotions, LAYERS.index(k), lineage="corpus"), window=8)
    for k in (6, 33, 51)
]
tp.layer_ternaries(per_layer, [f"layer {k}" for k in (6, 33, 51)], emotions).show()


In [6]:
tp.speed_figure(shard["speed"].astype(np.float32)[:, LAYERS.index(33)], starts).show()

**The confound check** — all 12 selfgen-lineage probes (plus the triple's corpus probes)
over the same story. The assigned emotion's row should dominate its own phase; any off-triple
row hot everywhere would mean the probes read valence/style, not emotion identity.

In [7]:
battery_idx = [i for i, name in enumerate(labels) if name.startswith("selfgen:")]
triple_idx = [labels.index(f"corpus:{e}") for e in emotions]
rows_idx = battery_idx + triple_idx
row_names = [labels[i] for i in rows_idx]
heat_by_layer = {}
for i, k in enumerate(LAYERS):
    d = shard["dots"].astype(np.float32)[:, i]
    d = d - d.mean(0, keepdims=True)
    nc = np.clip(shard["norms_centered"].astype(np.float32)[:, i][:, None], 1e-6, None)
    heat_by_layer[k] = d[:, rows_idx] / nc
trajectory_heatmap_scrubber(
    LAYERS, heat_by_layer, row_names, starts, default_layer=33,
    title="All probes over tokens — scrub layers",
).show()


**The animation** — the same trajectory as a playable sweep: press Play and watch the token
state tour the triangle phase by phase (every 2nd token, layer 33). This is the "video"
rendition of the phase-space view; the faint line is the full path, diamonds are phase starts.

In [8]:
from emotion_vectors.interactive import trajectory_ternary_animation

trajectory_ternary_animation(
    bary_by_layer[33], emotions, starts,
    title="Playable trajectory — layer 33 (softmax display transform, T=28)",
).show()


**The bridge to Q1** — the same tokens projected on the circumplex plane (PC1/PC2 of the 171
corpus-lineage emotion means at layer 33, the plane whose valence alignment H1 validated). The
projection is computed from the shard's stored probe dots alone (the PCA axes re-expressed as
weights over unit probes — exact, tested identity), so it needs no activations.

In [9]:
from emotion_vectors.trajectories import circumplex_weights

means_bundle = np.load(fetch("emotion_vectors_it_means.npz"), allow_pickle=True)
L33_FULL = list(means_bundle["layers"]).index(33)
weights = circumplex_weights(means_bundle["means"][:, L33_FULL].astype(np.float32))
corpus_idx = [i for i, name in enumerate(labels) if name.startswith("corpus:")]
d33 = shard["dots"].astype(np.float32)[:, LAYERS.index(33)][:, corpus_idx]
d33 = d33 - d33.mean(0, keepdims=True)
proj = tp.smooth((weights @ d33.T).T, window=8)
tp.circumplex_figure(proj, starts, emotions).show()


**The untransformed view** — raw centered cosines as 3D coordinates, no softmax. Interactive
only (rotate it); read positions and traversal order, not distances — the probe axes are
oblique in activation space.

In [10]:
tp.cosine_3d_figure(cos_by_layer[33], emotions, starts).show()

## 3. One simultaneous story, for contrast

A SIMULTANEOUS story should show all three emotions co-active — in the ternary view, a
trajectory hovering near the centroid rather than touring the corners.

In [11]:
sim = next(m for m in manifest if m["mode"] == "SIMULTANEOUS" and 150 < m["n_tokens"] < 260)
sshard = np.load(fetch(f"combined_trajectories/shards/{sim['story_id']}.npz"))
print(sim["story_id"], sim["emotions"])
sim_bary = {
    k: tp.barycentric(tp.smooth(tp.story_cosines(sshard, labels, sim["emotions"], i, lineage="corpus"), window=8))
    for i, k in enumerate(LAYERS)
}
trajectory_ternary_scrubber(
    LAYERS, sim_bary, sim["emotions"], [0], default_layer=33,
    title="SIMULTANEOUS story — should hover near the centroid; scrub layers",
).show()


t000_sim_p0_5bb39df3 ['unsettled', 'upset', 'cheerful']


## 4. Registered analysis — pending

The population verdict comes from the transition-locked average (all sequential transitions
aligned at t=0, incoming vs outgoing curves, bootstrap bands, per layer band) scored against
the random-direction null — `transition_windows` and `transition_locked_figure` are built and
tested, but the reads (ramp-vs-step statistic, anticipation window, pass bars) must be
registered in TREE.md **before** that figure is rendered over the corpus. This section will
hold the registered analysis and its verdict; until then, deliberately empty.